In [3]:
!pip install pytorch-tabnet autogluon -q
print("✅ 설치 완료")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 18.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 7.1 MB/s eta 0:00

In [4]:
import torch
print("GPU:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

from google.colab import drive
drive.mount('/content/drive')

GPU: True
Device: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ===============================================================
# 개선판 v2: 통합 정규화 + 모든 개선 + Hungarian 후처리
# ===============================================================
import os, random, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from scipy.optimize import linear_sum_assignment
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.metrics import Metric

# Config
SEED = 42
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 21
print(f"✅ Device: {DEVICE}")

DATA_DIR = "/content/drive/MyDrive/머신러닝/data"  # ⚠️ 본인 경로!
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
MODEL_DIR = os.path.join(DATA_DIR, "models_v2")
os.makedirs(MODEL_DIR, exist_ok=True)

# ===============================================================
# FE (그대로)
# ===============================================================
def feature_engineering(df):
    df = df.copy()
    if 'X_01' in df.columns:
        df = df.drop(columns=['X_01'])
    sensors = [c for c in df.columns if c.startswith('X_')]
    df['row_mean']   = df[sensors].mean(axis=1)
    df['row_std']    = df[sensors].std(axis=1)
    df['row_min']    = df[sensors].min(axis=1)
    df['row_max']    = df[sensors].max(axis=1)
    df['row_range']  = df['row_max'] - df['row_min']
    df['row_skew']   = df[sensors].skew(axis=1)
    df['row_median'] = df[sensors].median(axis=1)
    core_sensors = ['X_19', 'X_37', 'X_40', 'X_11']
    df['core_mean'] = df[core_sensors].mean(axis=1)
    df['core_std']  = df[core_sensors].std(axis=1)
    df['core_max']  = df[core_sensors].max(axis=1)
    df['core_min']  = df[core_sensors].min(axis=1)
    df['X19_X37_ratio'] = df['X_19'] / (df['X_37'] + 1e-6)
    df['X19_X40_ratio'] = df['X_19'] / (df['X_40'] + 1e-6)
    df['X19_X11_ratio'] = df['X_19'] / (df['X_11'] + 1e-6)
    df['X19_X37_diff']  = df['X_19'] - df['X_37']
    df['X19_X40_diff']  = df['X_19'] - df['X_40']
    df['X37_X40_diff']  = df['X_37'] - df['X_40']
    return df

# ===============================================================
# MacroF1
# ===============================================================
class MacroF1(Metric):
    def __init__(self):
        self._name = "macro_f1"
        self._maximize = True
    def __call__(self, y_true, y_score):
        return f1_score(y_true, np.argmax(y_score, axis=1), average='macro')

# ===============================================================
# Dataset, MLP, Losses (그대로)
# ===============================================================
class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = X.astype(np.float32)
        self.y = None if y is None else y.astype(np.int64)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        if self.y is None: return self.X[idx]
        return self.X[idx], self.y[idx]

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.2, act="relu"):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim); self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)
        self.act_fn = nn.ReLU(inplace=True) if act == "relu" else nn.GELU()
    def forward(self, x):
        residual = x
        out = self.act_fn(self.bn1(self.fc1(x)))
        out = self.dropout(out)
        out = self.bn2(self.fc2(out))
        return self.act_fn(out + residual)

class MLPEncoderWithHead(nn.Module):
    def __init__(self, input_dim, act="relu", num_classes=NUM_CLASSES):
        super().__init__()
        self.act_fn = nn.ReLU(inplace=True) if act == "relu" else nn.GELU()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), self.act_fn,
            ResidualBlock(256, act=act),
            nn.Linear(256, 256), nn.BatchNorm1d(256), self.act_fn,
            ResidualBlock(256, act=act)
        )
        self.embedding = nn.Linear(256, 128)
        self.projection = nn.Sequential(nn.Linear(128, 128), self.act_fn, nn.Linear(128, 128))
        self.classifier = nn.Sequential(nn.LayerNorm(128), self.act_fn, nn.Linear(128, num_classes))
    def forward(self, x):
        h = self.backbone(x)
        emb = self.embedding(h)
        z = self.projection(F.normalize(emb, dim=1))
        logits = self.classifier(emb)
        return emb, z, logits

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__(); self.temperature = temperature
    def forward(self, features, labels):
        device = features.device; bs = features.shape[0]
        labels = labels.contiguous().view(-1,1)
        mask = torch.eq(labels, labels.T).float().to(device)
        anchor_dot = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot, dim=1, keepdim=True)
        logits = anchor_dot - logits_max.detach()
        logits_mask = torch.ones_like(mask) - torch.eye(bs, device=device)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True)+1e-12)
        return -(mask * log_prob).sum(1).div(mask.sum(1)+1e-12).mean()

class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim, lambda_c=1.0):
        super().__init__()
        self.register_buffer('centers', torch.randn(num_classes, feat_dim))
        self.lambda_c = lambda_c
    def forward(self, emb, labels):
        return self.lambda_c * ((emb - self.centers[labels])**2).sum() / (2.0 * emb.size(0))
    @torch.no_grad()
    def update_centers(self, emb, labels, lr=0.5):
        for c in torch.unique(labels):
            mask = labels==c
            if mask.sum()==0: continue
            self.centers[c] = (1-lr)*self.centers[c] + lr*emb[mask].mean(dim=0)

def evaluate_model(model, loader):
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            _, _, logits = model(xb)
            preds.append(torch.softmax(logits, dim=1).cpu().numpy())
            trues.append(yb.cpu().numpy())
    preds, trues = np.vstack(preds), np.concatenate(trues)
    return f1_score(trues, preds.argmax(1), average="macro"), preds

def train_mlp_model(X, y, X_test, act="relu", epochs=100):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    test_preds = np.zeros((len(X_test), NUM_CLASSES))
    oof_preds = np.zeros((len(X), NUM_CLASSES))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔥 MLP-{act} Fold {fold}")
        train_loader = DataLoader(TabularDataset(X[tr_idx], y[tr_idx]),
                                  batch_size=256, shuffle=True, drop_last=True)
        val_loader = DataLoader(TabularDataset(X[val_idx], y[val_idx]),
                                batch_size=512, shuffle=False)
        test_loader = DataLoader(TabularDataset(X_test), batch_size=512, shuffle=False)

        model = MLPEncoderWithHead(input_dim=X.shape[1], act=act).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
        ce_loss = nn.CrossEntropyLoss(label_smoothing=0.05)
        supcon = SupConLoss(0.1)
        center = CenterLoss(NUM_CLASSES, 128, 1.0).to(DEVICE)

        best_f1, best_state, best_oof = -1, None, None
        for ep in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                emb, z, logits = model(xb)
                loss = 0.5*supcon(z,yb) + 1.0*ce_loss(logits,yb) + 0.1*center(emb,yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                center.update_centers(emb.detach(), yb.detach())
            f1, val_pred = evaluate_model(model, val_loader)
            scheduler.step(1-f1)
            if ep % 20 == 0: print(f"  Ep {ep}: f1={f1:.4f}")
            if f1 > best_f1:
                best_f1, best_state, best_oof = f1, model.state_dict(), val_pred

        print(f"  ✅ Best F1: {best_f1:.4f}")
        fold_scores.append(best_f1)
        oof_preds[val_idx] = best_oof
        model.load_state_dict(best_state)

        model.eval()
        all_test = []
        with torch.no_grad():
            for xb in test_loader:
                xb = xb.to(DEVICE)
                _, _, logits = model(xb)
                all_test.append(torch.softmax(logits, dim=1).cpu().numpy())
        test_preds += np.vstack(all_test) / 5

    print(f"\n📊 MLP-{act} 평균 OOF F1: {np.mean(fold_scores):.4f}")
    return test_preds, oof_preds

def train_tabnet(X, y, X_test):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    test_preds = np.zeros((len(X_test), NUM_CLASSES))
    oof_preds = np.zeros((len(X), NUM_CLASSES))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔥 TabNet Fold {fold}")
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        params = dict(
            n_d=32, n_a=16, n_steps=3, gamma=1.9, lambda_sparse=1e-5,
            optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=0.009),
            mask_type="sparsemax", seed=SEED, verbose=20,
            device_name='cuda' if torch.cuda.is_available() else 'cpu'
        )
        model = TabNetClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  eval_metric=[MacroF1],
                  patience=30, max_epochs=200, batch_size=1024)

        oof_pred = model.predict_proba(X_val)
        f1 = f1_score(y_val, oof_pred.argmax(1), average='macro')
        fold_scores.append(f1)
        print(f"  ✅ Fold {fold} F1: {f1:.4f}")
        oof_preds[val_idx] = oof_pred
        test_preds += model.predict_proba(X_test) / 5

    print(f"\n📊 TabNet 평균 OOF F1: {np.mean(fold_scores):.4f}")
    return test_preds, oof_preds

def train_autogluon(train_df, test_df):
    from autogluon.tabular import TabularPredictor
    predictor = TabularPredictor(
        label='target',
        eval_metric='f1_macro',
        path=os.path.join(MODEL_DIR, "autogluon"),
        problem_type='multiclass'
    ).fit(
        train_data=train_df,
        time_limit=3600,
        presets='best_quality',
        num_bag_folds=5,
        verbosity=2
    )
    return predictor.predict_proba(test_df).values

# ===============================================================
# ⭐ Hungarian 후처리 (이번엔 적용!)
# ===============================================================
def compute_quota(probs):
    N, C = probs.shape
    base = N // C; rem = N % C
    quota = np.full(C, base, int)
    order = np.argsort(-probs.mean(0))
    quota[order[:rem]] += 1
    return quota

def balanced_assign_hungarian(probs, quota):
    eps = 1e-15
    N, C = probs.shape
    cols, col_to_class = [], []
    for c in range(C):
        q = int(quota[c])
        cost_c = -np.log(probs[:,c] + eps)
        cols.append(np.repeat(cost_c[:,None], q, axis=1))
        col_to_class.extend([c]*q)
    cost = np.concatenate(cols, axis=1)
    row_ind, col_ind = linear_sum_assignment(cost)
    pred = np.zeros(N, int)
    pred[row_ind] = np.array(col_to_class)[col_ind]
    return pred

# ===============================================================
# Main
# ===============================================================
print("\n" + "="*60)
print("📊 데이터 로드 & FE")
print("="*60)
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train_fe = feature_engineering(train)
test_fe = feature_engineering(test)
print(f"FE 후: train {train_fe.shape}, test {test_fe.shape}")

X = train_fe.drop(columns=['ID', 'target']).values
y = train_fe['target'].values.astype(int)
X_test = test_fe.drop(columns=['ID']).values

# ⭐⭐⭐ 핵심 변경: 통합 정규화 ⭐⭐⭐
print("\n" + "="*60)
print("⭐ 통합 정규화 (train+test 합쳐서 fit)")
print("="*60)
print(f"Before - Train mean: {X.mean():.4f}, std: {X.std():.4f}")
print(f"Before - Test  mean: {X_test.mean():.4f}, std: {X_test.std():.4f}")

scaler = StandardScaler()
combined = np.vstack([X, X_test])  # ⭐ train+test 합치기
scaler.fit(combined)                # ⭐ 통합으로 fit
X = scaler.transform(X)
X_test = scaler.transform(X_test)

print(f"After  - Train mean: {X.mean():.4f}, std: {X.std():.4f}")
print(f"After  - Test  mean: {X_test.mean():.4f}, std: {X_test.std():.4f}")

# 학습
print("\n" + "="*60); print("🚀 1/4 TabNet"); print("="*60)
tab_test, tab_oof = train_tabnet(X, y, X_test)

print("\n" + "="*60); print("🚀 2/4 MLP-ReLU"); print("="*60)
relu_test, relu_oof = train_mlp_model(X, y, X_test, act="relu")

print("\n" + "="*60); print("🚀 3/4 MLP-GELU"); print("="*60)
gelu_test, gelu_oof = train_mlp_model(X, y, X_test, act="gelu")

print("\n" + "="*60); print("🚀 4/4 AutoGluon"); print("="*60)
fe_cols_no_target = [c for c in train_fe.columns if c not in ['ID', 'target']]
train_ag = pd.DataFrame(X, columns=fe_cols_no_target)
train_ag['target'] = y.astype(int)
test_ag = pd.DataFrame(X_test, columns=fe_cols_no_target)
ag_test = train_autogluon(train_ag, test_ag)

# 저장
np.save(f'{DATA_DIR}/v2_oof_tabnet.npy', tab_oof)
np.save(f'{DATA_DIR}/v2_oof_relu.npy', relu_oof)
np.save(f'{DATA_DIR}/v2_oof_gelu.npy', gelu_oof)
np.save(f'{DATA_DIR}/v2_test_tabnet.npy', tab_test)
np.save(f'{DATA_DIR}/v2_test_relu.npy', relu_test)
np.save(f'{DATA_DIR}/v2_test_gelu.npy', gelu_test)
np.save(f'{DATA_DIR}/v2_test_autogluon.npy', ag_test)
print("\n✅ v2 OOF & Test 저장 완료")

# OOF 점수
print("\n" + "="*60); print("📊 모델별 OOF F1"); print("="*60)
print(f"TabNet:   {f1_score(y, tab_oof.argmax(1), average='macro'):.4f}")
print(f"MLP-ReLU: {f1_score(y, relu_oof.argmax(1), average='macro'):.4f}")
print(f"MLP-GELU: {f1_score(y, gelu_oof.argmax(1), average='macro'):.4f}")

# ⭐ 앙상블 + Hungarian (이번엔!)
print("\n" + "="*60); print("⚖️ 4명 + Hungarian"); print("="*60)
ensemble = 0.25*tab_test + 0.25*relu_test + 0.25*gelu_test + 0.25*ag_test
quota = compute_quota(ensemble)
preds = balanced_assign_hungarian(ensemble, quota)

submission = pd.DataFrame({"ID": test["ID"], "target": preds})
output_path = os.path.join(DATA_DIR, "submission_v2_hungarian.csv")
submission.to_csv(output_path, index=False)
print(f"\n✅ Saved: {output_path}")
print(f"\n📊 클래스 분포:")
print(pd.Series(preds).value_counts().sort_index())

✅ Device: cuda

📊 데이터 로드 & FE
FE 후: train (21693, 70), test (15004, 69)

⭐ 통합 정규화 (train+test 합쳐서 fit)
Before - Train mean: 342.7570, std: 184446.2648
Before - Test  mean: 103.7252, std: 99096.7829
After  - Train mean: -0.0034, std: 1.0761
After  - Test  mean: 0.0049, std: 0.8784

🚀 1/4 TabNet

🔥 TabNet Fold 0


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 3.36461 | val_0_macro_f1: 0.08663 |  0:00:02s
epoch 20 | loss: 0.90241 | val_0_macro_f1: 0.64247 |  0:00:19s
epoch 40 | loss: 0.71496 | val_0_macro_f1: 0.71861 |  0:00:35s
epoch 60 | loss: 0.5587  | val_0_macro_f1: 0.76679 |  0:00:51s
epoch 80 | loss: 0.51786 | val_0_macro_f1: 0.7856  |  0:01:10s
epoch 100| loss: 0.43151 | val_0_macro_f1: 0.82114 |  0:01:26s
epoch 120| loss: 0.41316 | val_0_macro_f1: 0.80831 |  0:01:44s
epoch 140| loss: 0.37668 | val_0_macro_f1: 0.8008  |  0:02:01s
epoch 160| loss: 0.35496 | val_0_macro_f1: 0.8021  |  0:02:20s
epoch 180| loss: 0.34494 | val_0_macro_f1: 0.76755 |  0:02:40s
Stop training because you reached max_epochs = 200 with best_epoch = 191 and best_val_0_macro_f1 = 0.86021


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  ✅ Fold 0 F1: 0.8602

🔥 TabNet Fold 1


...

epoch 0  | loss: 3.31359 | val_0_macro_f1: 0.06717 |  0:00:00s
epoch 20 | loss: 1.04265 | val_0_macro_f1: 0.5752  |  0:00:17s
epoch 40 | loss: 0.73136 | val_0_macro_f1: 0.72753 |  0:00:33s
epoch 60 | loss: 0.55285 | val_0_macro_f1: 0.77977 |  0:00:50s
epoch 80 | loss: 0.44796 | val_0_macro_f1: 0.82375 |  0:01:07s
epoch 100| loss: 0.38699 | val_0_macro_f1: 0.81129 |  0:01:25s
epoch 120| loss: 0.37471 | val_0_macro_f1: 0.83419 |  0:01:41s

Early stopping occurred at epoch 135 with best_epoch = 105 and best_val_0_macro_f1 = 0.84377


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  ✅ Fold 4 F1: 0.8438

📊 TabNet 평균 OOF F1: 0.8512

🚀 2/4 MLP-ReLU

🔥 MLP-relu Fold 0
  Ep 0: f1=0.6171
  Ep 20: f1=0.8031
  Ep 40: f1=0.8206
  Ep 60: f1=0.8271
  Ep 80: f1=0.8299
  ✅ Best F1: 0.8312

🔥 MLP-relu Fold 1
  Ep 0: f1=0.5858
  Ep 20: f1=0.7956
  Ep 40: f1=0.8308
  Ep 60: f1=0.8643
  Ep 80: f1=0.8654
  ✅ Best F1: 0.8701

🔥 MLP-relu Fold 2
  Ep 0: f1=0.6004
  Ep 20: f1=0.8111
  Ep 40: f1=0.8583
  Ep 60: f1=0.8603
  Ep 80: f1=0.8610
  ✅ Best F1: 0.8671

🔥 MLP-relu Fold 3
  Ep 0: f1=0.5697
  Ep 20: f1=0.7980
  Ep 40: f1=0.8265
  Ep 60: f1=0.8644
  Ep 80: f1=0.8661
  ✅ Best F1: 0.8711

🔥 MLP-relu Fold 4
  Ep 0: f1=0.6092
  Ep 20: f1=0.7833
  Ep 40: f1=0.8044
  Ep 60: f1=0.8225
  Ep 80: f1=0.8152
  ✅ Best F1: 0.8230

📊 MLP-relu 평균 OOF F1: 0.8525

🚀 3/4 MLP-GELU

🔥 MLP-gelu Fold 0
  Ep 0: f1=0.6019
  Ep 20: f1=0.8307
  Ep 40: f1=0.8502
  Ep 60: f1=0.8751
  Ep 80: f1=0.8752
  ✅ Best F1: 0.8785

🔥 MLP-gelu Fold 1
  Ep 0: f1=0.5815
  Ep 20: f1=0.8147
  Ep 40: f1=0.8520
  Ep 60: f1=0.8

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.10.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.55/14.56 GB
Total GPU Memory:   Free: 14.55 GB, Allocated: 0.02 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       9.91 GB / 12.67 GB (78.2%)
Disk Space Avail:   61.62 GB / 112.64 GB (54.7%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked


✅ v2 OOF & Test 저장 완료

📊 모델별 OOF F1
TabNet:   0.8533
MLP-ReLU: 0.8526
MLP-GELU: 0.8630

⚖️ 4명 + Hungarian

✅ Saved: /content/drive/MyDrive/머신러닝/data/submission_v2_hungarian.csv

📊 클래스 분포:
0     715
1     714
2     714
3     715
4     715
5     714
6     714
7     714
8     715
9     715
10    714
11    714
12    715
13    715
14    714
15    715
16    714
17    715
18    714
19    714
20    715
Name: count, dtype: int64


In [5]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

DATA_DIR = '/content/drive/MyDrive/머신러닝/data'

# v2 데이터 로드
tab_test = np.load(f'{DATA_DIR}/v2_test_tabnet.npy')
relu_test = np.load(f'{DATA_DIR}/v2_test_relu.npy')
gelu_test = np.load(f'{DATA_DIR}/v2_test_gelu.npy')
ag_test = np.load(f'{DATA_DIR}/v2_test_autogluon.npy')

test = pd.read_csv(f'{DATA_DIR}/test.csv')

# Hungarian 함수
def compute_quota(probs):
    N, C = probs.shape
    base = N // C; rem = N % C
    quota = np.full(C, base, int)
    order = np.argsort(-probs.mean(0))
    quota[order[:rem]] += 1
    return quota

def balanced_assign_hungarian(probs, quota):
    eps = 1e-15
    N, C = probs.shape
    cols, col_to_class = [], []
    for c in range(C):
        q = int(quota[c])
        cost_c = -np.log(probs[:,c] + eps)
        cols.append(np.repeat(cost_c[:,None], q, axis=1))
        col_to_class.extend([c]*q)
    cost = np.concatenate(cols, axis=1)
    row_ind, col_ind = linear_sum_assignment(cost)
    pred = np.zeros(N, int)
    pred[row_ind] = np.array(col_to_class)[col_ind]
    return pred

# ============================================
# 여러 조합 만들기
# ============================================
combos = [
    ("a_gelu_ag_up", 0.20, 0.20, 0.30, 0.30),
    ("b_ag_up",      0.20, 0.20, 0.25, 0.35),
    ("c_tab_down",   0.15, 0.25, 0.30, 0.30),
    ("d_3models",    1/3, 1/3, 1/3, 0.0),
    ("e_ag_strong",  0.15, 0.20, 0.25, 0.40),
]

for name, w_tab, w_relu, w_gelu, w_ag in combos:
    ensemble = w_tab*tab_test + w_relu*relu_test + w_gelu*gelu_test + w_ag*ag_test
    quota = compute_quota(ensemble)
    preds = balanced_assign_hungarian(ensemble, quota)

    submission = pd.DataFrame({"ID": test["ID"], "target": preds})
    output_path = f'{DATA_DIR}/submission_v2_{name}.csv'
    submission.to_csv(output_path, index=False)
    print(f"✅ {name}: tab={w_tab} relu={w_relu} gelu={w_gelu} ag={w_ag}")

print("\n총 5개 파일 생성!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/머신러닝/data/v2_test_tabnet.npy'